# RK1828 Image Classification Demo (MobileNetV2 / ResNet50-v2)

This notebook runs ImageNet classification on the **RK1828 AI coprocessor**
(PCIe-attached, 20 TOPS NPU) and walks through the full inference pipeline:

| Step | Content |
|---|---|
| 1 | Download models (from huggingface.co) |
| 2 | Pick a model and an image |
| 3 | Pre-processing (turn the image into a tensor) |
| 4 | RKNN inference (on the NPU) |
| 5 | Post-processing (dequantize + Softmax, Top-5) |
| 6 | Multi-model comparison (2 models x 2 quantizations) |

> Background: **i8** quantizes the weights to 8-bit integers (smaller, faster);
> **fp16** is the 16-bit float version. Run both and compare.

## 0. Environment check

Make sure the RKNN3 runtime (`librknn3_api.so`) and the RK1828 device are ready.
If this fails, install the driver/runtime first (the `rknn3-rk182x` DEB package)
and check with `sudo rknn-smi info` that the RK1828 shows up.

In [ ]:
from rknn3 import get_lib, find_devices

_ = get_lib()  # loads /usr/lib/librknn3_api.so; failure means the runtime is not installed

devices = find_devices()
if not devices:
    raise RuntimeError("No RK1828 device found! Check: sudo rknn-smi info")
for dev_id, dev_type in devices:
    print(f"Found device: id={dev_id}  interface={dev_type}")

## 1. Download models

The models for this demo are hosted on Hugging Face (account `qhddz`) and are
downloaded directly from **huggingface.co**. Run the next cell to start.

> Files that already exist (and are non-empty) are **skipped automatically** —
> re-running never downloads them twice.
> If your network cannot reach huggingface.co, set up a proxy first, e.g.
> `export https_proxy=http://<your-proxy>:<port>` before starting Jupyter.


This demo downloads MobileNetV2 and ResNet50-v2 (i8 and fp16 each), plus the ImageNet labels and a sample image.

In [ ]:
# (HF repo, path in repo, local path)
FILES = [
    ("mobilenet_v2-rknn3-rk1828", "i8/mobilenet_v2.rknn", "models/cnn/mobilenet_v2/i8/mobilenet_v2.rknn"),
    ("mobilenet_v2-rknn3-rk1828", "i8/mobilenet_v2.weight", "models/cnn/mobilenet_v2/i8/mobilenet_v2.weight"),
    ("mobilenet_v2-rknn3-rk1828", "fp16/mobilenet_v2.rknn", "models/cnn/mobilenet_v2/fp16/mobilenet_v2.rknn"),
    ("mobilenet_v2-rknn3-rk1828", "fp16/mobilenet_v2.weight", "models/cnn/mobilenet_v2/fp16/mobilenet_v2.weight"),
    ("mobilenet_v2-rknn3-rk1828", "imagenet_labels.txt", "datasets/imagenet_labels.txt"),
    ("mobilenet_v2-rknn3-rk1828", "dog_224x224.jpg", "datasets/dog_224x224.jpg"),
    ("resnet50_v2-rknn3-rk1828", "i8/resnet50_v2.rknn", "models/cnn/resnet50_v2/i8/resnet50_v2.rknn"),
    ("resnet50_v2-rknn3-rk1828", "i8/resnet50_v2.weight", "models/cnn/resnet50_v2/i8/resnet50_v2.weight"),
    ("resnet50_v2-rknn3-rk1828", "fp16/resnet50_v2.rknn", "models/cnn/resnet50_v2/fp16/resnet50_v2.rknn"),
    ("resnet50_v2-rknn3-rk1828", "fp16/resnet50_v2.weight", "models/cnn/resnet50_v2/fp16/resnet50_v2.weight"),
    ("resnet50_v2-rknn3-rk1828", "imagenet_labels.txt", "datasets/imagenet_labels.txt"),
    ("resnet50_v2-rknn3-rk1828", "dog_224x224.jpg", "datasets/dog_224x224.jpg"),
]

In [ ]:
from nb_download import ensure_files

ensure_files(FILES, "hf")

## 2. Pick a model and an image

- **Model**: `mobilenet_v2` (lightweight, 3.4M params) or `resnet50_v2` (classic, 25M params)
- **Quantization**: `i8` (int8) or `fp16` (half precision)
- **Image**: samples under `datasets/` (you can drop your own ~224x224 image there)

In [ ]:
import glob
import ipywidgets as widgets
from IPython.display import display

w_model = widgets.Dropdown(options=["mobilenet_v2", "resnet50_v2"],
                           description="Model:")
w_quant = widgets.Dropdown(options=["i8", "fp16"], description="Quant:")
w_image = widgets.Dropdown(options=sorted(glob.glob("datasets/*.jpg")),
                           description="Image:", layout=widgets.Layout(width="60%"))
display(w_model, w_quant, w_image)

LABELS = open("datasets/imagenet_labels.txt").read().splitlines()
print(f"{len(LABELS)} classes. Then keep running downwards ▼")

## 3. Pre-processing

The model takes **224x224 NHWC uint8 RGB** (normalization is baked into the
model — no mean/std needed). We resize the image to 224x224 and add a batch dim.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from vision_post import setup_cjk_font

setup_cjk_font()
img_orig = Image.open(w_image.value).convert("RGB")
img = img_orig.resize((224, 224))
x = np.asarray(img)[None]  # [1,224,224,3] uint8 NHWC
print("input tensor:", x.shape, x.dtype)

# show every pre-processing stage: original -> resize -> input tensor
fig, axes = plt.subplots(1, 3, figsize=(11, 3.8))
for ax, im, title in [
    (axes[0], img_orig, f"1. original {img_orig.size[0]}x{img_orig.size[1]}"),
    (axes[1], img, "2. resize 224x224"),
    (axes[2], x[0], f"3. model input {x.shape}\nuint8 NHWC"),
]:
    ax.imshow(np.asarray(im))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. RKNN inference

Load the `.rknn` (compute graph) + `.weight` (weights) onto the RK1828 and run
one timed inference.

In [ ]:
import time
from rknn3 import Rknn3Model

model_path = f"models/cnn/{w_model.value}/{w_quant.value}/{w_model.value}.rknn"
model = Rknn3Model(model_path)
print("input attrs:", model.input_attrs)
print("output attrs:", model.output_attrs)

_ = model.infer([x])  # warmup
t0 = time.time()
outputs = model.infer([x])
print(f"inference time: {(time.time()-t0)*1000:.1f} ms")

## 5. Post-processing: dequantize + Softmax

The i8 model outputs int8; restore float logits with the output tensor's own
`scale` / `zero_point`, then Softmax and take the Top-5.

In [ ]:
from vision_post import dequant

attr = model.output_attrs[0]
logits = dequant(outputs[0], attr).ravel()
probs = np.exp(logits - logits.max())
probs /= probs.sum()

top5 = probs.argsort()[::-1][:5]
for i in top5:
    print(f"{probs[i]*100:5.1f}%  {LABELS[i]}")

In [ ]:
import matplotlib.pyplot as plt
from vision_post import setup_cjk_font

setup_cjk_font()
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), gridspec_kw={"width_ratios": [1, 1.6]})
axes[0].imshow(img)
axes[0].set_title(f"{w_model.value} ({w_quant.value})")
axes[0].axis("off")
names = [LABELS[i] for i in top5][::-1]
axes[1].barh(names, [probs[i] for i in top5][::-1], color="#3b82f6")
axes[1].set_title("Top-5 probability")
axes[1].set_xlim(0, 1)
plt.tight_layout()
plt.show()

model.close()

## 6. Multi-model comparison (i8 + fp16, full sweep)

Same image, running **2 models x 2 quantizations (i8 / fp16) = 4 combos**.
Progress is printed live; afterwards you get a comparison table, a latency
bar chart, and a Top-5 mosaic. Each model is timed 3 times and the best is
kept (warm-up jitter excluded).

In [ ]:
import time
from rknn3 import Rknn3Model
from vision_post import dequant

ALL_MODELS = ["mobilenet_v2", "resnet50_v2"]
QUANTS = ["i8", "fp16"]

compare = []  # (model, quant, latency_ms, top5_idx, top5_prob)
total = len(ALL_MODELS) * len(QUANTS)
step = 0
for name in ALL_MODELS:
    for quant in QUANTS:
        step += 1
        print(f"[{step}/{total}] loading {name} ({quant}) ...", flush=True)
        m = Rknn3Model(f"models/cnn/{name}/{quant}/{name}.rknn")
        m.infer([x])  # warmup
        best, outs = float("inf"), None
        for _ in range(3):
            t0 = time.time()
            outs = m.infer([x])
            best = min(best, (time.time() - t0) * 1000)
        logits = dequant(outs[0], m.output_attrs[0]).ravel()
        p = np.exp(logits - logits.max())
        p /= p.sum()
        top5_i = p.argsort()[::-1][:5]
        compare.append((name, quant, best, top5_i, p[top5_i]))
        m.close()
        print(f"      -> {best:6.1f} ms, Top-1: {LABELS[top5_i[0]]} "
              f"({p[top5_i[0]]*100:.1f}%)", flush=True)
print("\nAll done ✅", flush=True)

### Comparison table

In [ ]:
from IPython.display import HTML

rows = "".join(
    f"<tr><td>{n}</td><td>{q}</td><td>{lat:.1f}</td>"
    f"<td>{', '.join(f'{LABELS[i]}({pp*100:.1f}%)' for i, pp in list(zip(t, pr))[:5])}</td></tr>"
    for n, q, lat, t, pr in compare)
HTML("<table style='text-align:left'>"
     "<tr><th>Model</th><th>Quant</th><th>Latency (ms)</th><th>Top-5 (prob)</th></tr>"
     + rows + "</table>")

### Latency bar chart (i8 vs fp16)

In [ ]:
import matplotlib.pyplot as plt
from vision_post import setup_cjk_font

setup_cjk_font()
colors = {"i8": "#3b82f6", "fp16": "#f59e0b"}
xs = np.arange(len(ALL_MODELS))
width = 0.38
fig, ax = plt.subplots(figsize=(8, 4.5))
for k, q in enumerate(QUANTS):
    lats = [lat for n, qq, lat, *_ in compare if qq == q]
    tops = [LABELS[t[0]] for n, qq, _, t, _ in compare if qq == q]
    bars = ax.bar(xs + (k - 0.5) * width, lats, width, label=q, color=colors[q])
    for bar, lat, top in zip(bars, lats, tops):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                f"{lat:.0f}ms\n{top}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(xs, ALL_MODELS)
ax.set_ylabel("inference latency (ms)")
ax.set_title("Classification models (same image, i8 vs fp16)")
ax.set_ylim(0, max(lat for _, _, lat, *_ in compare) * 1.35)
ax.legend()
plt.tight_layout()
plt.show()

### Top-5 mosaic

In [ ]:
import matplotlib.pyplot as plt
from vision_post import setup_cjk_font

setup_cjk_font()
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
for ax, (n, q, lat, t, pr) in zip(axes.flat, compare):
    names = [LABELS[i] for i in t][::-1]
    ax.barh(names, list(pr)[::-1], color="#3b82f6" if q == "i8" else "#f59e0b")
    ax.set_xlim(0, 1)
    ax.set_title(f"{n} ({q})  {lat:.0f}ms", fontsize=11)
plt.tight_layout()
plt.show()

## Summary

- Full pipeline: **download -> pre-process -> NPU inference -> dequantize/Softmax -> Top-5**
- What step 6 shows:
  - **mobilenet_v2** (lightweight, 3.4M params) is clearly faster than
    **resnet50_v2** (classic, 25M params); ResNet50-v2 is more accurate
    (ImageNet Top-1 ~72% vs ~75.6%, paper figures).
  - For the same model, i8 is usually faster than fp16 with nearly identical
    Top-5 — quantization costs very little accuracy.